# Breaking the Feedback Trap: Complete Runnable Version
Phiên bản này sẽ tự động quét thư mục `/kaggle/input` để tìm dataset thực tế (dựa trên cấu trúc thư mục `images` và `masks` như các notebook cũ). Nếu không tìm thấy, nó sẽ dùng Synthetic Data để đảm bảo luôn chạy mượt mà 100%.

## Cell 1: Setup & Environment

In [ ]:
!pip install -q segmentation-models-pytorch albumentations
import os
import cv2
import glob
import time
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.backends.cudnn.deterministic = True

seed_everything(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
os.makedirs("outputs", exist_ok=True)

## Cell 2: Data Pipeline (Auto-detect Real Datasets)

In [ ]:
class MedicalSegmentationDataset(Dataset):
    def __init__(self, image_paths=None, mask_paths=None, transform=None, num_synthetic=100):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.transform = transform
        self.use_synthetic = (image_paths is None or len(image_paths) == 0)
        self.num_synthetic = num_synthetic

    def __len__(self):
        return self.num_synthetic if self.use_synthetic else len(self.image_paths)

    def __getitem__(self, idx):
        if self.use_synthetic:
            image = np.random.randint(50, 200, (256, 256, 3), dtype=np.uint8)
            mask = np.zeros((256, 256), dtype=np.float32)
            cv2.circle(mask, (128 + random.randint(-20,20), 128 + random.randint(-20,20)), random.randint(30, 60), 1, -1)
        else:
            image = cv2.imread(self.image_paths[idx])
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            mask = cv2.imread(self.mask_paths[idx], cv2.IMREAD_GRAYSCALE)
            mask = (mask > 127).astype(np.float32)

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']
        
        mask = mask.unsqueeze(0)
        return image, mask

train_transform = A.Compose([
    A.Resize(256, 256),
    A.HorizontalFlip(p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(256, 256),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

# TỰ ĐỘNG TÌM DATASET (Hành vi từ các notebook cũ Kvasir/ISIC)
SRC = None
for root, dirs, _ in os.walk("/kaggle/input"):
    if "images" in dirs and "masks" in dirs:
        SRC = root
        break

train_img_paths, train_mask_paths = [], []
val_img_paths, val_mask_paths = [], []

if SRC is not None:
    print(f"✅ Found real dataset at: {SRC}")
    all_imgs = sorted(glob.glob(os.path.join(SRC, 'images', '*.*')))
    all_masks = sorted(glob.glob(os.path.join(SRC, 'masks', '*.*')))
    
    # Simple 80/20 split
    split_idx = int(len(all_imgs) * 0.8)
    train_img_paths, val_img_paths = all_imgs[:split_idx], all_imgs[split_idx:]
    train_mask_paths, val_mask_paths = all_masks[:split_idx], all_masks[split_idx:]
    print(f"  -> Train: {len(train_img_paths)}, Val: {len(val_img_paths)}")
else:
    print("⚠️ No real dataset found in /kaggle/input (Missing images/masks folders).")
    print("  -> Using Synthetic Data for demonstration.")

train_dataset = MedicalSegmentationDataset(train_img_paths, train_mask_paths, transform=train_transform, num_synthetic=160)
val_dataset = MedicalSegmentationDataset(val_img_paths, val_mask_paths, transform=val_transform, num_synthetic=40)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)
print("Data Pipeline Ready!")

## Cell 3: Mechanistic Models & Loss

In [ ]:
class TverskyLoss(nn.Module):
    def __init__(self, alpha=0.3, beta=0.7, smooth=1e-6):
        super().__init__()
        self.alpha, self.beta, self.smooth = alpha, beta, smooth

    def forward(self, inputs, targets):
        inputs = torch.sigmoid(inputs).view(-1)
        targets = targets.view(-1)
        TP = (inputs * targets).sum()
        FP = ((1 - targets) * inputs).sum()
        FN = (targets * (1 - inputs)).sum()
        return 1 - ((TP + self.smooth) / (TP + self.alpha * FN + self.beta * FP + self.smooth))

class MiniRecurrentNet(nn.Module):
    def __init__(self, mode='baseline'):
        super().__init__()
        self.mode = mode 
        self.enc = nn.Sequential(nn.Conv2d(4, 32, 3, padding=1), nn.ReLU(), nn.Conv2d(32, 32, 3, padding=1), nn.ReLU())
        self.dec = nn.Conv2d(32, 1, 3, padding=1)
        
    def forward(self, x, iters=2):
        B, C, H, W = x.shape
        m_prev = torch.zeros(B, 1, H, W, device=x.device)
        
        for t in range(iters):
            inp = torch.cat([x, m_prev], dim=1)
            feat = self.enc(inp)
            f_guidance = torch.sigmoid(self.dec(feat)) 
            
            if self.mode == 'baseline':
                m_prev = f_guidance 
            elif self.mode == 'proposed':
                m_prev = 1 - (1 - f_guidance.detach()) * (1 - m_prev)
                
        return m_prev 

model_base = MiniRecurrentNet(mode='baseline').to(device)
model_prop = MiniRecurrentNet(mode='proposed').to(device)

criterion = TverskyLoss(alpha=0.3, beta=0.7)
opt_base = torch.optim.AdamW(model_base.parameters(), lr=1e-3)
opt_prop = torch.optim.AdamW(model_prop.parameters(), lr=1e-3)
print("Models, Loss, and Optimizers initialized.")

## Cell 4: The Training Loop

In [ ]:
def calc_metrics(preds, masks, thresh=0.5):
    preds_bin = (preds > thresh).float()
    masks = masks.float()
    TP = (preds_bin * masks).sum().item()
    FP = (preds_bin * (1 - masks)).sum().item()
    FN = ((1 - preds_bin) * masks).sum().item()
    TN = ((1 - preds_bin) * (1 - masks)).sum().item()
    dice = (2 * TP) / (2 * TP + FP + FN + 1e-6)
    fpr = FP / (FP + TN + 1e-6)
    return dice, fpr

def train_model(model, optimizer, name="Model", num_epochs=5):
    history = {'train_loss': [], 'val_loss': [], 'val_dice': [], 'val_fpr': []}
    
    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0
        for imgs, masks in train_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            optimizer.zero_grad()
            out = model(imgs)
            loss = criterion(out, masks)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            
        model.eval()
        val_loss, val_dice, val_fpr = 0.0, 0.0, 0.0
        with torch.no_grad():
            for imgs, masks in val_loader:
                imgs, masks = imgs.to(device), masks.to(device)
                out = model(imgs)
                loss = criterion(out, masks)
                dice, fpr = calc_metrics(out, masks)
                val_loss += loss.item()
                val_dice += dice
                val_fpr += fpr
                
        history['train_loss'].append(train_loss / len(train_loader))
        history['val_loss'].append(val_loss / len(val_loader))
        history['val_dice'].append(val_dice / len(val_loader))
        history['val_fpr'].append(val_fpr / len(val_loader))
        
        print(f"[{name}] Ep {epoch+1}/{num_epochs} | Val Dice: {history['val_dice'][-1]:.4f} | Val FPR: {history['val_fpr'][-1]:.4f}")
    
    pd.DataFrame(history).to_csv(f"outputs/{name}_history.csv", index=False)
    torch.save(model.state_dict(), f"outputs/{name}.pth")
    return history

print("Training Baseline (Hard Feedback)...")
hist_base = train_model(model_base, opt_base, name="Baseline", num_epochs=5)

print("\nTraining Proposed (Detached Soft-OR)...")
hist_prop = train_model(model_prop, opt_prop, name="Proposed", num_epochs=5)

## Cell 5: Quantitative & Qualitative Visualization

In [ ]:
def plot_comparison(h_base, h_prop):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    epochs = range(1, len(h_base['val_dice']) + 1)
    
    ax1.plot(epochs, h_base['val_dice'], 'r--', label='Baseline Dice')
    ax1.plot(epochs, h_prop['val_dice'], 'g-', label='Proposed Dice (Ours)')
    ax1.set_title('Validation Dice Comparison')
    ax1.legend()
    
    ax2.plot(epochs, h_base['val_fpr'], 'r--', label='Baseline FPR (Trap)')
    ax2.plot(epochs, h_prop['val_fpr'], 'g-', label='Proposed FPR (Firewall)')
    ax2.set_title('Validation FPR Comparison (Lower is Better)')
    ax2.legend()
    
    plt.savefig("outputs/learning_curves.png")
    plt.show()

def visualize_feedback_trap(model_b, model_p, dataloader):
    model_b.eval()
    model_p.eval()
    imgs, masks = next(iter(dataloader))
    
    with torch.no_grad():
        out_b = (model_b(imgs.to(device)).cpu() > 0.5).float()
        out_p = (model_p(imgs.to(device)).cpu() > 0.5).float()
    
    img_np = imgs[0].permute(1, 2, 0).numpy()
    img_np = img_np * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
    img_np = np.clip(img_np, 0, 1)
    gt = masks[0, 0].numpy()
    
    def get_overlay(img, truth, pred):
        overlay = img.copy()
        tp = (truth == 1) & (pred == 1)
        fp = (truth == 0) & (pred == 1)
        overlay[tp] = overlay[tp] * 0.3 + np.array([0, 1, 0]) * 0.7 
        overlay[fp] = overlay[fp] * 0.3 + np.array([1, 0, 0]) * 0.7 
        return np.clip(overlay, 0, 1)
    
    fig, axs = plt.subplots(1, 4, figsize=(20, 5))
    axs[0].imshow(img_np); axs[0].set_title("Input Image")
    axs[1].imshow(gt, cmap='gray'); axs[1].set_title("Ground Truth")
    axs[2].imshow(get_overlay(img_np, gt, out_b[0,0].numpy())); axs[2].set_title("Baseline (Red = Feedback Trap)")
    axs[3].imshow(get_overlay(img_np, gt, out_p[0,0].numpy())); axs[3].set_title("Detached Soft-OR (Firewall)")
    for ax in axs: ax.axis('off')
    
    plt.savefig("outputs/feedback_trap_visualization.png")
    plt.show()

print("Plotting quantitative results...")
plot_comparison(hist_base, hist_prop)
print("Plotting qualitative Feedback Trap overlay...")
visualize_feedback_trap(model_base, model_prop, val_loader)
print("✅ TẤT CẢ FILE OUTPUT ĐÃ ĐƯỢC LƯU VÀO THƯ MỤC /outputs/")